In [1]:
import pandas as pd
import numpy as np

In [2]:
ruta_input = "../data/processed/"
ruta_output = "../data/processed/"

In [3]:
def cargar_archivos():
    items = pd.read_csv(ruta_input + "order_items_clean.csv")
    products = pd.read_csv(ruta_input + "products_clean.csv")
    orders = pd.read_csv(ruta_input + "orders_clean.csv")
    users = pd.read_csv(ruta_input + "users_clean.csv")
    distribution_centers = pd.read_csv(ruta_input + "distribution_centers_clean.csv")
    return items, products, orders, users, distribution_centers

Merges

In [4]:
def crear_base_unificada(items, products, orders, users, distribution_centers):
    print("--- INICIO DE UNIFICACIÓN ---")
    
    # 1. Unimos Items con Productos
    # Añadimos 'distribution_center_id' para poder conectar con la tabla de logística después
    df = pd.merge(items, 
                  products[['id', 'cost', 'category', 'brand', 'distribution_center_id']], 
                  left_on='product_id', right_on='id', how='left')
    
    # Al unir product_id con id, se crea una columna 'id' que no queremos
    if 'id' in df.columns:
        df = df.drop('id', axis=1)

    # 2. Unimos con Centros de Distribución (Hoja C: Logística)
    df = pd.merge(df, 
                  distribution_centers[['id', 'name']], 
                  left_on='distribution_center_id', right_on='id', how='left')
    
    # Limpiamos el 'id' que viene de distribution_centers y renombramos 'name'
    if 'id' in df.columns:
        df = df.drop('id', axis=1)
    df = df.rename(columns={'name': 'dist_center_name'})

    # 3. Unimos con Órdenes
    df = pd.merge(df, 
                  orders[['order_id', 'status', 'num_of_item']], 
                  on='order_id', how='left')

    # 4. Unimos con Usuarios
    df = pd.merge(df, 
                  users[['id', 'country', 'gender', 'age']], 
                  left_on='user_id', right_on='id', how='left')
    
    # Limpiamos el 'id' que viene de users
    if 'id' in df.columns:
        df = df.drop('id', axis=1)

    print("--- UNIFICACIÓN COMPLETADA CON ÉXITO ---")
    return df

CREACIÓN DE COLUMNAS ESTRATÉGICAS Y DATAFRAME FINAL

In [5]:
def tabla_principal_ventas(df):
    # A. Finanzas: Margen Neto
    df['margin'] = df['sale_price'] - df['cost']
    df['margin_percentage'] = (df['margin'] / df['sale_price']) * 100
    
    # B. Logística: Días de envío
    df['shipped_at'] = pd.to_datetime(df['shipped_at'])
    df['delivered_at'] = pd.to_datetime(df['delivered_at'])
    df['is_returned'] = df['status_x'].apply(lambda x: 1 if x == 'Returned' else 0)
    
    # C. Producto: Estacionalidad
    df['created_at'] = pd.to_datetime(df['created_at'])
    df['mes'] = df['created_at'].dt.month
    df['year'] = df['created_at'].dt.year
    
    def asignar_estacion(mes):
        if mes in [12, 1, 2]: return 'Invierno'
        elif mes in [3, 4, 5]: return 'Primavera'
        elif mes in [6, 7, 8]: return 'Verano'
        else: return 'Otoño'
    
    df['estacion'] = df['mes'].apply(asignar_estacion)

    # D. Comportamiento del Cliente
    # 0 = Primera Compra, 1 = Compra Recurrente
    df['es_recurrente'] = df.duplicated(subset=['user_id'], keep='first').astype(int)
   
    print(f"✅ Tabla principal ventas creada con {df.shape[0]} filas y {df.shape[1]} columnas.")
    df.to_csv(ruta_output + "tabla_principal_ventas.csv", index=False)
    
    return df

if __name__ == "__main__":
    items, products, orders, users, distribution_centers = cargar_archivos()
    df_final = crear_base_unificada(items, products, orders, users, distribution_centers)

    print(df_final.columns)
    df_final = tabla_principal_ventas(df_final)

--- INICIO DE UNIFICACIÓN ---
--- UNIFICACIÓN COMPLETADA CON ÉXITO ---
Index(['id_x', 'order_id', 'user_id', 'product_id', 'inventory_item_id',
       'status_x', 'created_at', 'shipped_at', 'delivered_at', 'returned_at',
       'sale_price', 'id_y', 'cost', 'category', 'brand',
       'distribution_center_id', 'dist_center_name', 'status_y', 'num_of_item',
       'country', 'gender', 'age'],
      dtype='object')
✅ Tabla principal ventas creada con 181759 filas y 29 columnas.


In [6]:
def buscar_columnas_duplicadas(df):
    print(" Analizando si hay columnas con contenido idéntico...")
    duplicadas = []
    columnas = df.columns

    for i in range(len(columnas)):     
        for j in range(i + 1, len(columnas)):
            col2 = columnas[j]
            col1 = columnas[i]

            if col2 not in duplicadas:
                if df[col1].equals(df[col2]):
                    duplicadas.append(col2)
                    print(f"  Detectadas: '{col1}' y '{col2}' tienen el mismo contenido.")
                    
    if not duplicadas:
        print(" Todo limpio: No hay columnas con contenido repetido.")
    
    return duplicadas

if __name__ == "__main__":
    resultados = buscar_columnas_duplicadas(df_final)

 Analizando si hay columnas con contenido idéntico...
  Detectadas: 'product_id' y 'id_y' tienen el mismo contenido.
  Detectadas: 'status_x' y 'status_y' tienen el mismo contenido.


In [7]:
df_final = df_final.drop(columns=['status_y'], errors='ignore')
df_final = df_final.drop(columns=['id_y'], errors='ignore')

df_final = df_final.rename(columns={
    'status_x': 'status',
    'id_x': 'item_id',
    'sale_price': 'revenue'
})

print(" Columnas actuales en memoria:")
print(df_final.columns)

df_final.to_csv("../data/processed/tabla_principal_ventas.csv", index=False)

 Columnas actuales en memoria:
Index(['item_id', 'order_id', 'user_id', 'product_id', 'inventory_item_id',
       'status', 'created_at', 'shipped_at', 'delivered_at', 'returned_at',
       'revenue', 'cost', 'category', 'brand', 'distribution_center_id',
       'dist_center_name', 'num_of_item', 'country', 'gender', 'age', 'margin',
       'margin_percentage', 'is_returned', 'mes', 'year', 'es_recurrente',
       'estacion'],
      dtype='object')
